# 🫀 Heart Murmur Detection with LSTM

**Full Pipeline — Google Colab (GPU / T4)**

| Step | Stage |
|---|---|
| ⚙️ 1 | Install dependencies |
| 🔐 2 | Load secrets & env vars (Colab Secrets) |
| 🏗️ 3 | GPU verification |
| 📁 4 | Project structure |
| 📝 5 | Write module files (`config`, `logger`, `preprocessing`, `visualizations`, `model_loader`) |
| 🤗 6 | Download LSTM model from Hugging Face |
| 🔊 7 | Audio upload, loading & MFCC extraction |
| 🧠 8 | Inference & prediction |
| 📊 9 | MLflow tracking + artifact logging (DagsHub) |
| 📦 10 | Model registration & stage transition |
| 📈 11 | Summary & artifact listing |

---
## ⚙️ Step 1 — Install Dependencies

In [ ]:
%%capture install_log
!pip install librosa soundfile
!pip install tensorflow[and-cuda]
!pip install huggingface_hub
!pip install mlflow dagshub
!pip install matplotlib seaborn

In [ ]:
import importlib
REQUIRED = ['librosa', 'tensorflow', 'mlflow', 'huggingface_hub', 'matplotlib', 'seaborn']
missing  = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    raise EnvironmentError(f'Missing packages: {missing} — re-run Step 1.')
print('✅ All dependencies available.')

---
## 🔐 Step 2 — Environment Variables (Colab Secrets)

In [ ]:
import os
from google.colab import userdata

# ── MongoDB ───────────────────────────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ updated

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — logs saved inside Colab
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

# ── TensorFlow GPU optimisation ───────────────────────────────────────────────
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

---
## 🏗️ Step 3 — GPU Verification

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)  # avoid OOM
    print(f'✅ GPU detected: {[g.name for g in gpus]}')
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU.')

print(f'   TensorFlow version : {tf.__version__}')
!nvidia-smi | head -15

---
## 📁 Step 4 — Project Structure

In [ ]:
import os, sys

ROOT_DIR      = os.getcwd()                          # /content
AUDIO_DIR     = os.path.join(ROOT_DIR, 'audio')
MODEL_DIR     = os.path.join(ROOT_DIR, 'model')
UTILS_DIR     = os.path.join(ROOT_DIR, 'utils')
UI_DIR        = os.path.join(ROOT_DIR, 'ui')
ARTIFACTS_DIR = os.path.join(ROOT_DIR, 'artifacts')
PLOTS_DIR     = os.path.join(ARTIFACTS_DIR, 'plots')
FEATURES_DIR  = os.path.join(ARTIFACTS_DIR, 'features')

for d in [AUDIO_DIR, MODEL_DIR, UTILS_DIR, UI_DIR, ARTIFACTS_DIR, PLOTS_DIR, FEATURES_DIR]:
    os.makedirs(d, exist_ok=True)
    init_path = os.path.join(d, '__init__.py')
    if not os.path.exists(init_path):
        open(init_path, 'w').close()

open(os.path.join(ROOT_DIR, '__init__.py'), 'w').close()

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print('✅ Project structure ready.')
for d in [ROOT_DIR, AUDIO_DIR, MODEL_DIR, UTILS_DIR, UI_DIR, ARTIFACTS_DIR]:
    print(f'   {d}')

---
## 📝 Step 5 — Write Module Files

Recreates all source modules on disk — mirrors the original project layout exactly.

In [ ]:
# ── config.py ─────────────────────────────────────────────────────────────────
config_code = '''
import os

# TensorFlow optimisation flag
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Audio config
SAMPLE_RATE = 22050
N_MFCC      = 52

# Hugging Face model config
HF_REPO_ID        = "Diveshj/heart-murmur"
HF_MODEL_FILENAME = "lstm_model.keras"
'''.strip()

with open(os.path.join(ROOT_DIR, 'config.py'), 'w') as f:
    f.write(config_code)
print('✅ config.py')

In [ ]:
# ── utils/logger.py ───────────────────────────────────────────────────────────
logger_code = '''
import logging

def setup_logger(name: str):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler   = logging.StreamHandler()
        formatter = logging.Formatter(
            "[%(asctime)s] [%(levelname)s] %(name)s - %(message)s"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger
'''.strip()

with open(os.path.join(UTILS_DIR, 'logger.py'), 'w') as f:
    f.write(logger_code)
print('✅ utils/logger.py')

In [ ]:
# ── audio/preprocessing.py ────────────────────────────────────────────────────
preprocessing_code = '''
import librosa
import numpy as np
from config import SAMPLE_RATE, N_MFCC
from utils.logger import setup_logger

logger = setup_logger("AudioPreprocessing")


def load_audio(uploaded_file):
    try:
        logger.info("Loading audio file")
        y, sr = librosa.load(uploaded_file, sr=SAMPLE_RATE)
        return y, sr
    except Exception as e:
        logger.exception("Audio loading failed")
        raise RuntimeError("Invalid or corrupted audio file") from e


def extract_mfcc(y, sr):
    try:
        logger.info("Extracting MFCC features")
        mfcc        = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        mfcc_scaled = np.mean(mfcc.T, axis=0)
        X_input     = np.expand_dims(mfcc_scaled, axis=0)
        X_input     = np.expand_dims(X_input,     axis=2)
        return X_input
    except Exception as e:
        logger.exception("MFCC extraction failed")
        raise RuntimeError("Feature extraction failed") from e
'''.strip()

with open(os.path.join(AUDIO_DIR, 'preprocessing.py'), 'w') as f:
    f.write(preprocessing_code)
print('✅ audio/preprocessing.py')

In [ ]:
# ── ui/visualizations.py ──────────────────────────────────────────────────────
viz_code = '''
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt


def plot_waveform(y, sr, title="Waveform"):
    fig, ax = plt.subplots(figsize=(12, 3))
    librosa.display.waveshow(y, sr=sr, ax=ax, color="steelblue")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    plt.tight_layout()
    return fig


def plot_mfcc(mfcc, sr, title="MFCC"):
    fig, ax = plt.subplots(figsize=(12, 4))
    img = librosa.display.specshow(mfcc, sr=sr, x_axis="time", ax=ax, cmap="viridis")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    return fig


def plot_prediction_bar(prediction, class_labels=None):
    scores = prediction[0]
    if class_labels is None:
        class_labels = [f"Class {i}" for i in range(len(scores))]
    colors = ["#e74c3c" if i == scores.argmax() else "#3498db" for i in range(len(scores))]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(class_labels, scores, color=colors, edgecolor="black", linewidth=0.6)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Confidence")
    ax.set_title("Prediction Scores", fontsize=13)
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
    plt.tight_layout()
    return fig
'''.strip()

with open(os.path.join(UI_DIR, 'visualizations.py'), 'w') as f:
    f.write(viz_code)
print('✅ ui/visualizations.py')

In [ ]:
# ── model/model_loader.py ─────────────────────────────────────────────────────
model_loader_code = '''
import os
from huggingface_hub import hf_hub_download
from tensorflow import keras
from config import HF_REPO_ID, HF_MODEL_FILENAME
from utils.logger import setup_logger

logger = setup_logger("ModelLoader")


def load_model(local_model_dir: str = "model"):
    """Download model from HuggingFace Hub (cached) and load with Keras."""
    try:
        local_path = os.path.join(local_model_dir, HF_MODEL_FILENAME)
        if not os.path.exists(local_path):
            logger.info(f"Downloading model: {HF_REPO_ID}/{HF_MODEL_FILENAME}")
            hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_MODEL_FILENAME,
                local_dir=local_model_dir
            )
            logger.info(f"Model downloaded to: {local_path}")
        else:
            logger.info(f"Using cached model: {local_path}")
        model = keras.models.load_model(local_path)
        logger.info("Model loaded successfully")
        return model
    except Exception as e:
        logger.exception("Model loading failed")
        raise RuntimeError("Could not load LSTM model") from e
'''.strip()

with open(os.path.join(MODEL_DIR, 'model_loader.py'), 'w') as f:
    f.write(model_loader_code)
print('✅ model/model_loader.py')

# ── Verify file tree ──────────────────────────────────────────────────────────
print('\n📂 Module file tree:')
SKIP = {'.ipynb_checkpoints', '__pycache__', 'mlruns', 'artifacts', 'sample_data'}
for root, dirs, files in os.walk(ROOT_DIR):
    dirs[:] = sorted(d for d in dirs if d not in SKIP and not d.startswith('.'))
    level   = root.replace(ROOT_DIR, '').count(os.sep)
    indent  = '   ' * level
    print(f'{indent}{os.path.basename(root) or "/content"}/')
    for file in sorted(files):
        if file.endswith('.py'):
            print(f'{indent}   {file}')

---
## 🤗 Step 6 — Download LSTM Model from Hugging Face

In [ ]:
# Force-reload modules after writing them to disk
import importlib
import utils.logger, audio.preprocessing, ui.visualizations, model.model_loader
for mod in [utils.logger, audio.preprocessing, ui.visualizations, model.model_loader]:
    importlib.reload(mod)

from model.model_loader import load_model
from utils.logger       import setup_logger

logger = setup_logger('HeartMurmurPipeline')

model = load_model(local_model_dir=MODEL_DIR)

print('\n✅ Model loaded!')
print(f'   Input shape  : {model.input_shape}')
print(f'   Output shape : {model.output_shape}')
print(f'   Parameters   : {model.count_params():,}')
model.summary()

---
## 🔊 Step 7 — Audio Upload, Loading & MFCC Extraction

In [ ]:
from google.colab import files as colab_files

print('📂 Upload a heart sound file (.wav or .mp3):')
uploaded = colab_files.upload()

In [ ]:
import numpy as np
import librosa
import matplotlib
matplotlib.use('Agg')   # stable non-interactive backend for Colab
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from audio.preprocessing import load_audio, extract_mfcc
from ui.visualizations   import plot_waveform, plot_mfcc, plot_prediction_bar

if not uploaded:
    raise FileNotFoundError('No file uploaded — re-run cell 7a.')

AUDIO_FILENAME = list(uploaded.keys())[0]
AUDIO_PATH     = os.path.join(ROOT_DIR, AUDIO_FILENAME)
print(f'✅ File: {AUDIO_FILENAME}')

# Load audio
y, sr    = load_audio(AUDIO_PATH)
duration = librosa.get_duration(y=y, sr=sr)

print(f'\n🎵 Audio info:')
print(f'   Sample rate : {sr} Hz')
print(f'   Duration    : {duration:.2f} s')
print(f'   Samples     : {len(y):,}')

# Play inline
display(Audio(data=y, rate=sr))

# Waveform plot
fig_wave  = plot_waveform(y, sr, title=f'Waveform — {AUDIO_FILENAME}')
wave_path = os.path.join(PLOTS_DIR, 'waveform.png')
fig_wave.savefig(wave_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'   Saved → {wave_path}')

In [ ]:
from config import N_MFCC

# Extract MFCC features (model input)
X_input = extract_mfcc(y, sr)

print(f'✅ MFCC extracted — X_input shape: {X_input.shape}  (batch, n_mfcc, 1)')

# Full MFCC spectrogram for visualisation
mfcc_full = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
fig_mfcc  = plot_mfcc(mfcc_full, sr, title=f'MFCC — {AUDIO_FILENAME}')
mfcc_path = os.path.join(PLOTS_DIR, 'mfcc.png')
fig_mfcc.savefig(mfcc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'   Saved → {mfcc_path}')

# Save raw features
features_path = os.path.join(FEATURES_DIR, 'mfcc_features.npy')
np.save(features_path, X_input)
print(f'   Features → {features_path}')

---
## 🧠 Step 8 — Inference & Prediction

In [ ]:
# Adjust class labels to match your model's training classes
CLASS_LABELS = {
    0: 'Normal',
    1: 'Murmur',
    2: 'Extrasystole'
}

logger.info('Running inference on GPU...')
prediction    = model.predict(X_input, verbose=0)
predicted_idx = int(np.argmax(prediction, axis=1)[0])
confidence    = float(prediction[0][predicted_idx])
predicted_lbl = CLASS_LABELS.get(predicted_idx, f'Class {predicted_idx}')

print('\n🔮 Prediction Results')
print('─' * 40)
print(f'   Predicted class : {predicted_idx} — {predicted_lbl}')
print(f'   Confidence      : {confidence:.4f} ({confidence*100:.2f}%)')
print(f'   Raw scores      : {np.round(prediction[0], 4)}')

# Bar chart
fig_pred       = plot_prediction_bar(prediction, class_labels=list(CLASS_LABELS.values()))
pred_plot_path = os.path.join(PLOTS_DIR, 'prediction_scores.png')
fig_pred.savefig(pred_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'   Saved → {pred_plot_path}')

logger.info(f'Inference complete — {predicted_lbl} ({confidence*100:.2f}%)')

---
## 📊 Step 9 — MLflow Experiment Tracking (DagsHub)

In [ ]:
import mlflow
import mlflow.keras
import json

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment('heart-murmur-detection')

EXPERIMENT_INFO_PATH = os.path.join(ARTIFACTS_DIR, 'experiment_info.json')

print(f'✅ MLflow configured.')
print(f'   Tracking URI : {mlflow.get_tracking_uri()}')
print(f'   Experiment   : heart-murmur-detection')

In [ ]:
from config import SAMPLE_RATE, N_MFCC, HF_REPO_ID, HF_MODEL_FILENAME

with mlflow.start_run(run_name='lstm_inference_run') as run:

    RUN_ID = run.info.run_id
    print(f'🚀 MLflow Run ID: {RUN_ID}')

    # ── Parameters ────────────────────────────────────────────────────────────
    mlflow.log_params({
        'sample_rate'       : SAMPLE_RATE,
        'n_mfcc'            : N_MFCC,
        'hf_repo_id'        : HF_REPO_ID,
        'hf_model_filename' : HF_MODEL_FILENAME,
        'audio_file'        : AUDIO_FILENAME,
        'audio_duration_s'  : round(duration, 4),
        'model_total_params': model.count_params(),
        'model_input_shape' : str(model.input_shape),
        'model_output_shape': str(model.output_shape),
        'predicted_class'   : predicted_idx,
        'predicted_label'   : predicted_lbl,
        'use_gpu'           : len(tf.config.list_physical_devices('GPU')) > 0
    })
    print('   ✅ Parameters logged')

    # ── Metrics ───────────────────────────────────────────────────────────────
    mlflow.log_metric('confidence', confidence)
    for idx, score in enumerate(prediction[0]):
        lbl = CLASS_LABELS.get(idx, f'class_{idx}')
        mlflow.log_metric(f'score_{lbl}', float(score))
    print('   ✅ Metrics logged')

    # ── Plot artifacts ────────────────────────────────────────────────────────
    mlflow.log_artifact(wave_path,      artifact_path='plots')
    mlflow.log_artifact(mfcc_path,      artifact_path='plots')
    mlflow.log_artifact(pred_plot_path, artifact_path='plots')
    print('   ✅ Plots logged')

    # ── Feature artifacts ─────────────────────────────────────────────────────
    mlflow.log_artifact(features_path, artifact_path='features')
    print('   ✅ MFCC features logged')

    # ── Model summary as text artifact ────────────────────────────────────────
    summary_path = os.path.join(ARTIFACTS_DIR, 'model_summary.txt')
    with open(summary_path, 'w') as f:
        model.summary(print_fn=lambda x: f.write(x + '\n'))
    mlflow.log_artifact(summary_path, artifact_path='model_info')
    print('   ✅ Model summary logged')

    # ── Log Keras model (also auto-registers) ─────────────────────────────────
    mlflow.keras.log_model(
        model,
        artifact_path='lstm_model',
        registered_model_name='heart_murmur_lstm'
    )
    print('   ✅ Keras model logged & registered as heart_murmur_lstm')

    # ── Tags ──────────────────────────────────────────────────────────────────
    mlflow.set_tags({
        'model_type'   : 'LSTM',
        'task'         : 'Heart Murmur Detection',
        'framework'    : 'TensorFlow/Keras',
        'model_source' : 'HuggingFace Hub',
        'audio_format' : AUDIO_FILENAME.rsplit('.', 1)[-1].upper()
    })

    # ── Save experiment_info.json (mirrors register_model.py pattern) ─────────
    artifact_uri   = mlflow.get_artifact_uri()
    model_uri      = f'{artifact_uri}/lstm_model'
    experiment_info = {
        'run_id'       : RUN_ID,
        'artifact_uri' : artifact_uri,
        'model_uri'    : model_uri,
        'tracking_uri' : mlflow.get_tracking_uri(),
        'experiment'   : 'heart-murmur-detection'
    }
    with open(EXPERIMENT_INFO_PATH, 'w') as f:
        json.dump(experiment_info, f, indent=4)
    mlflow.log_artifact(EXPERIMENT_INFO_PATH, artifact_path='meta')
    print('   ✅ experiment_info.json saved & logged')

print(f'\n✅ MLflow run complete!')
print(f'   Run ID       : {RUN_ID}')
print(f'   Artifact URI : {artifact_uri}')
print(f'   Model URI    : {model_uri}')

---
## 📦 Step 10 — Model Registration & Stage Transition

In [ ]:
from mlflow.tracking import MlflowClient

client     = MlflowClient()
MODEL_NAME = 'heart_murmur_lstm'

try:
    versions = client.get_latest_versions(MODEL_NAME)
    if versions:
        latest = versions[-1]
        print(f'✅ Latest registered version : {latest.version}  (stage: {latest.current_stage})')

        client.transition_model_version_stage(
            name=MODEL_NAME,
            version=latest.version,
            stage='Staging',
            archive_existing_versions=True
        )
        print(f'   ✅ Version {latest.version} → Staging')
    else:
        print('⚠️  No registered versions found — Step 9 log_model handles registration.')
except Exception as e:
    print(f'⚠️  Registration note: {e}')
    print('   (Expected if tracking server does not support the Model Registry.)')

---
## 📈 Step 11 — Summary & Artifact Listing

In [ ]:
import json

with open(EXPERIMENT_INFO_PATH) as f:
    exp_info = json.load(f)

print('=' * 62)
print('  🫀  HEART MURMUR DETECTION — PIPELINE COMPLETE')
print('=' * 62)
print(f'  🎵  Audio file        : {AUDIO_FILENAME}')
print(f'  ⏱️   Duration          : {duration:.2f} s')
print(f'  🔢  MFCC input shape  : {X_input.shape}')
print(f'  🧠  Prediction        : {predicted_lbl} (class {predicted_idx})')
print(f'  📊  Confidence        : {confidence*100:.2f}%')
print(f'  🚀  MLflow Run ID     : {exp_info["run_id"]}')
print(f'  🌐  Tracking URI      : {exp_info["tracking_uri"]}')
print(f'  📦  Model URI         : {exp_info["model_uri"]}')
print(f'  📋  Registry name     : heart_murmur_lstm  →  Staging')
print('=' * 62)

print('\n📁 Local artifacts:')
SKIP = {'.ipynb_checkpoints', '__pycache__'}
for root, dirs, files in os.walk(ARTIFACTS_DIR):
    dirs[:] = sorted(d for d in dirs if d not in SKIP)
    level   = root.replace(ARTIFACTS_DIR, '').count(os.sep)
    indent  = '   ' * (level + 1)
    print(f'{indent}{os.path.basename(root)}/')
    for file in sorted(files):
        size = os.path.getsize(os.path.join(root, file))
        print(f'{indent}   {file}  ({size:,} bytes)')

In [ ]:
import shutil

zip_base = os.path.join(ROOT_DIR, 'heart_murmur_artifacts')
shutil.make_archive(zip_base, 'zip', ARTIFACTS_DIR)
zip_file = zip_base + '.zip'

print(f'✅ All artifacts zipped → {zip_file}')
print(f'   Size: {os.path.getsize(zip_file):,} bytes')

# Uncomment to download to your local machine:
# from google.colab import files as colab_files
# colab_files.download(zip_file)